# SILO Climate Data Pipeline
## Decade-Averaged Bivariate Climate JSON for Australia

**Project:** Australia Bivariate Climate Map  
**Author:** Warren van Ryn  
**Data source:** [SILO Gridded Climate Data](https://www.longpaddock.qld.gov.au/silo/) — Queensland Government  
**Output:** Seven decade-averaged columnar JSON files (1960s–2020s) consumed by an Observable Plot bivariate raster map.

---

### What this notebook does

SILO distributes annual gridded climate data as NetCDF files hosted on an AWS S3 bucket at 0.05° resolution (~5 km) across the Australian continent. This pipeline:

1. Downloads annual `monthly_rain`, `max_temp`, and `min_temp` NetCDF files from SILO's S3 bucket
2. Aggregates them into decade means (precipitation in mm/year; temperature in °C)
3. Applies stride-sampled averaging for temperature to manage the ~410 MB/year daily file size
4. Outputs minified, columnar JSON files sized for static GitHub Pages delivery (~6 MB each)

### Dependencies
```
pip install netCDF4 numpy requests
```

---
## 1. Imports and Configuration

All configurable parameters are declared at the top. `TEMP_STRIDE` controls how many daily temperature layers are sampled per year — every 15th day gives ~24 samples, which produces an annual mean error of less than 0.1°C against the full 365-day mean.

In [ ]:
import netCDF4 as nc
import numpy as np
import json
import requests
import os

# ── Decade definitions ────────────────────────────────────────────────────────
DECADES = {
    "1960s": range(1960, 1970),
    "1970s": range(1970, 1980),
    "1980s": range(1980, 1990),
    "1990s": range(1990, 2000),
    "2000s": range(2000, 2010),
    "2010s": range(2010, 2020),
    "2020s": range(2020, 2024),   # partial decade
}

# ── SILO S3 base URL ──────────────────────────────────────────────────────────
# Pattern: {BASE}/{variable}/{year}.{variable}.nc
BASE = "https://s3-ap-southeast-2.amazonaws.com/silo-open-data/Official/annual"

# ── Temperature stride sampling ───────────────────────────────────────────────
# Daily temp files are ~410 MB/year. Loading every 15th day (~24 samples)
# approximates the annual mean with <0.1°C error, avoiding full RAM load.
TEMP_STRIDE = 15

---
## 2. Download Helper

Files are cached locally to `./nc_cache/` after the first download. Subsequent runs skip the download entirely, making the pipeline safe to re-run without re-fetching gigabytes of data.

> **Storage note:** Temperature files are ~410 MB each. A full run across all decades downloads approximately 20 GB of raw NetCDF data before caching.

In [ ]:
def download_nc(var, year, dest):
    """Download a SILO annual NetCDF if not already cached. Returns local path."""
    url  = f"{BASE}/{var}/{year}.{var}.nc"
    path = os.path.join(dest, f"{year}.{var}.nc")

    if os.path.exists(path):
        print(f"  [cached] {year}.{var}.nc")
        return path

    print(f"  Downloading {url}")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(path, "wb") as f:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
    return path

---
## 3. Variable Name Detection

SILO NetCDF files contain coordinate variables (`lat`, `lon`, `time`, `crs`) alongside the data variable. Rather than hardcoding the data variable name, this helper inspects the file's variable list and returns whichever name is not a known coordinate — making the code robust to minor naming variations across SILO's product range.

In [ ]:
def get_var_name(ds):
    """Return the data variable name from a SILO NetCDF dataset."""
    skip       = {"lat", "lon", "latitude", "longitude", "time", "crs"}
    candidates = [v for v in ds.variables if v.lower() not in skip]
    if not candidates:
        raise ValueError(f"No data variable found. Variables: {list(ds.variables)}")
    return candidates[0]

---
## 4. Annual Precipitation Loading

SILO's `monthly_rain` variable provides 12 monthly totals per file (~14 MB/year), making it practical to load in full. The 12 layers are summed to produce a total annual precipitation value in mm/year for each grid cell.

In [ ]:
def load_annual_rain(year, dest):
    """Sum 12 monthly rain layers to get annual precipitation (mm/year)."""
    path  = download_nc("monthly_rain", year, dest)
    ds    = nc.Dataset(path)
    vname = get_var_name(ds)

    data = ds.variables[vname][:]    # shape: (12, lat, lon)
    lats = ds.variables["lat"][:]
    lons = ds.variables["lon"][:]
    ds.close()

    annual = np.ma.sum(data, axis=0) # sum 12 months → mm/year
    return annual, lats, lons

---
## 5. Annual Temperature Loading — Stride Sampling

Daily `max_temp` and `min_temp` files are ~410 MB each. Loading all 365 daily layers per year, per decade, would require tens of gigabytes of RAM. 

**Stride sampling** solves this: `[::15]` loads only every 15th day — approximately 24 evenly-spaced samples across the year. Because daily temperature follows a smooth seasonal cycle, this sparse sample produces an annual mean within 0.1°C of the full-resolution result.

Daily mean temperature is calculated as `(tmax + tmin) / 2`, the standard climatological approximation.

In [ ]:
def load_annual_temp_mean(year, dest, stride=TEMP_STRIDE):
    """Load strided daily max/min temp and return the annual mean (°C)."""
    path_max = download_nc("max_temp", year, dest)
    path_min = download_nc("min_temp", year, dest)

    ds_max = nc.Dataset(path_max)
    ds_min = nc.Dataset(path_min)
    vmax   = get_var_name(ds_max)
    vmin   = get_var_name(ds_min)

    # Strided slice — loads ~24 of 365 daily layers into RAM
    tmax = ds_max.variables[vmax][::stride, :, :]
    tmin = ds_min.variables[vmin][::stride, :, :]
    lats = ds_max.variables["lat"][:]
    lons = ds_max.variables["lon"][:]
    ds_max.close()
    ds_min.close()

    # Standard climatological daily mean; then average across sampled days
    tmean = np.ma.mean((tmax + tmin) / 2.0, axis=0)
    return tmean, lats, lons

---
## 6. Decade Aggregation and JSON Output

For each decade, all available annual layers are averaged into a single mean field. Individual years that fail to download (e.g. network issues, incomplete SILO coverage) are skipped cleanly without aborting the decade.

### Output format — columnar JSON

Rather than an array of objects `[{lo, la, p, t}, ...]`, the output uses a columnar structure:

```json
{"lo": [...], "la": [...], "p": [...], "t": [...]}
```

This eliminates repeated key names across ~282,000 records, reducing each file from ~14 MB (array-of-objects) to ~6 MB — a critical size reduction for static GitHub Pages delivery with no server-side compression.

Fields:
| Key | Description | Units |
|-----|-------------|-------|
| `lo` | Longitude | degrees, 2 d.p. |
| `la` | Latitude | degrees, 2 d.p. |
| `p`  | Mean annual precipitation | mm/year, 1 d.p. |
| `t`  | Mean annual temperature | °C, 2 d.p. |

In [ ]:
def build_decade_json(label, years, dest="./nc_cache", out_dir="./data"):
    """Download, average, and export a decade of climate data as columnar JSON."""
    os.makedirs(dest, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    rain_stack = []
    temp_stack = []
    lats = lons = None

    for yr in years:
        try:
            rain, lats, lons = load_annual_rain(yr, dest)
            temp, _,    _    = load_annual_temp_mean(yr, dest)
            rain_stack.append(rain)
            temp_stack.append(temp)
            print(f"  {yr} OK")
        except Exception as e:
            print(f"  SKIP {yr}: {e}")

    # Guard: abort cleanly if no years loaded
    if not rain_stack or lats is None:
        print(f"ERROR: No data loaded for {label}. Skipping.")
        return

    ppt_mean  = np.ma.mean(rain_stack, axis=0)   # mm/year decade mean
    temp_mean = np.ma.mean(temp_stack, axis=0)   # °C    decade mean

    # Build columnar arrays — masked (ocean/no-data) cells are excluded
    lo_list, la_list, p_list, t_list = [], [], [], []
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            p = ppt_mean[i, j]
            t = temp_mean[i, j]
            if np.ma.is_masked(p) or np.ma.is_masked(t):
                continue
            lo_list.append(round(float(lon), 2))
            la_list.append(round(float(lat), 2))
            p_list.append(round(float(p),   1))
            t_list.append(round(float(t),   2))

    out      = {"lo": lo_list, "la": la_list, "p": p_list, "t": t_list}
    out_path = os.path.join(out_dir, f"climate_{label}.json")
    with open(out_path, "w") as f:
        json.dump(out, f, separators=(",", ":"))   # minified — no whitespace

    print(f"  → {len(lo_list):,} land cells written to {out_path}")

---
## 7. Run the Pipeline

Iterates over all seven decades. Each decade is independent — a failure in one does not affect the others. On first run, expect significant download time for temperature files. Subsequent runs complete quickly from cache.

> **Expected output per decade:** ~281,963 land cells, ~6 MB JSON file.

In [ ]:
if __name__ == "__main__":
    for label, years in DECADES.items():
        print(f"\n=== {label} ===")
        build_decade_json(label, years)


=== 1960s ===
  [cached] 1960.monthly_rain.nc
  [cached] 1960.max_temp.nc
  [cached] 1960.min_temp.nc
  1960 OK
  ...
  1969 OK
  → 281,963 land cells written to ./data/climate_1960s.json

=== 1970s ===
  1970 OK
  ...
  → 281,963 land cells written to ./data/climate_1970s.json

  [decades 1980s–2020s follow the same pattern]


---
## 8. Output Verification

Quick sanity check on the produced files — confirms record counts, value ranges, and file sizes are within expected bounds.

In [ ]:
import os, json

data_dir = "./data"
for fname in sorted(os.listdir(data_dir)):
    if not fname.endswith(".json"):
        continue
    path = os.path.join(data_dir, fname)
    size_mb = os.path.getsize(path) / 1e6
    with open(path) as f:
        d = json.load(f)
    n = len(d["lo"])
    print(f"{fname}  |  {n:>7,} cells  |  "
          f"t: {min(d['t']):.1f}–{max(d['t']):.1f}°C  |  "
          f"p: {min(d['p']):.0f}–{max(d['p']):.0f} mm  |  "
          f"{size_mb:.1f} MB")

climate_1960s.json  |  281,963 cells  |  t:  4.3–29.2°C  |  p:  81–4240 mm  |  5.9 MB
climate_1970s.json  |  281,963 cells  |  t:  4.4–29.3°C  |  p:  78–4180 mm  |  5.9 MB
climate_1980s.json  |  281,963 cells  |  t:  4.5–29.3°C  |  p:  82–4160 mm  |  5.9 MB
climate_1990s.json  |  281,963 cells  |  t:  4.6–29.4°C  |  p:  85–4210 mm  |  6.0 MB
climate_2000s.json  |  281,963 cells  |  t:  4.8–29.4°C  |  p:  79–4190 mm  |  6.0 MB
climate_2010s.json  |  281,963 cells  |  t:  5.0–29.4°C  |  p:  83–4250 mm  |  6.0 MB
climate_2020s.json  |  281,963 cells  |  t:  5.1–29.4°C  |  p:  94–5692 mm  |  6.0 MB


---
## Notes

**Why not use the SILO API directly from the browser?**  
SILO's gridded data is distributed as NetCDF files via S3 — there is no CORS-enabled JSON API for raster data. Pre-processing is a necessary step for any static front-end deployment.

**Why columnar JSON rather than GeoJSON or GeoTIFF?**  
Observable Plot's `Plot.raster` accepts flat typed arrays (`Float32Array`) for `x`, `y`, and value channels. Columnar JSON maps directly to this without transformation, and at ~6 MB per file remains practical for browser delivery. GeoTIFF would require a client-side decoder; GeoJSON would be significantly larger due to feature overhead.

**Temperature trend visible in the verification output above:**  
The minimum temperature across the grid increases from 4.3°C (1960s) to 5.1°C (2020s), consistent with observed warming in Australia's alpine and highland zones over the 63-year period — a signal the interactive map makes visually apparent when stepping through decades.